<a href="https://colab.research.google.com/github/gsmagrandi-svg/FreeConvert_Universel_Grandiose/blob/main/Free_Convert_Universel_Grandiose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 FreeConvert_Universel_Grandiose

## 📝 Description
**FreeConvert_Universel_Grandiose** est un convertisseur de fichiers universel basé sur Gradio. Il prend en charge plus de **1500 conversions** couvrant les vidéos, images, fichiers audio, documents, livres numériques et archives. L'interface détecte automatiquement le type de fichier importé et propose les formats de destination ainsi que les options avancées (résolution, qualité, bitrate, etc.).

## 🎯 Objectifs
- Convertir n'importe quel fichier vers un autre format.
- Fonctionner sur tous les appareils (Windows, Mac, Linux, Mobile) via un navigateur web.
- Offrir une expérience utilisateur professionnelle avec barres de progression et instructions claires.
- Sauvegarder les fichiers convertis soit sur le PC local, soit sur Google Drive.

## 🛠️ Étapes de configuration
1. **Montage du Google Drive** : Connexion au Drive pour stocker les fichiers de sortie.
2. **Installation des dépendances système** : FFmpeg (vidéo/audio), LibreOffice (documents), Calibre (ebooks), ImageMagick (images), polices arabes.
3. **Installation des packages Python** : Gradio, WeasyPrint, PyPDF, BeautifulSoup.
4. **Définition des formats supportés** : Dictionnaire `FORMATS_SUPPORTES` listant les extensions, formats de sortie et options par catégorie.
5. **Fonctions de conversion** : Modules dédiés pour chaque type de fichier (vidéo, audio, image, document, ebook, archive).
6. **Interface Gradio** : Détection automatique, menus dynamiques, barre de progression, sauvegarde PC/Drive.
7. **Lancement de l'application** : `demo.launch(share=True)`.

## 📂 Chemin de sauvegarde sur Drive
`/content/drive/MyDrive/FreeConvert_Universel_Grandiose`

## ⚠️ Note
La première exécution peut prendre plusieurs minutes (installation de FFmpeg, LibreOffice, Calibre). Les exécutions suivantes seront plus rapides grâce au cache de Colab.

In [10]:
# ==============================================================================
# PARTIE 1 : CONFIGURATION PERSISTANTE (GOOGLE DRIVE & DÉPENDANCES)
# ==============================================================================
import os
import sys
import subprocess
import time
import tempfile
import shutil
import html
import json
from pathlib import Path

# --- Montage du Google Drive ---
print("🔗 Connexion au Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# --- Chemins du projet ---
PROJECT_DIR = "/content/drive/MyDrive/FreeConvert_Universel_Grandiose"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"📁 Dossier projet configuré sur : {PROJECT_DIR}")

# --- ÉTAPE 1 : NETTOYAGE DU DOSSIER CORROMPU SUR LE DRIVE ---
PACKAGES_DIR = os.path.join(PROJECT_DIR, "pip_packages")
if os.path.exists(PACKAGES_DIR):
    print("🧹 Suppression du dossier 'pip_packages' corrompu sur le Drive...")
    shutil.rmtree(PACKAGES_DIR)
    print("✅ Dossier corrompu supprimé.")
else:
    print("✅ Aucun dossier 'pip_packages' corrompu trouvé sur le Drive.")

# --- ÉTAPE 2 : RETRAIT DU CHEMIN DE sys.path ---
if PACKAGES_DIR in sys.path:
    sys.path.remove(PACKAGES_DIR)
    print("✅ Chemin 'pip_packages' retiré de sys.path.")

# --- ÉTAPE 3 : RÉINSTALLATION FORCÉE DE GRADIO DANS L'ENVIRONNEMENT NATIF ---
print("⏳ Réinstallation forcée de Gradio dans l'environnement Colab...")
!pip install -q --upgrade --force-reinstall gradio
print("✅ Gradio réinstallé proprement.")

# --- ÉTAPE 4 : INSTALLATION DES AUTRES PACKAGES MANQUANTS ---
print("⏳ Installation des packages Python complémentaires...")
!pip install -q weasyprint pypdf beautifulsoup4 pillow requests
print("✅ Packages complémentaires installés.")

# --- ÉTAPE 5 : INSTALLATION DES OUTILS SYSTÈME ---
print("🔧 Installation des outils système (peut prendre quelques minutes)...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq ffmpeg libreoffice calibre imagemagick \
    libpango-1.0-0 libharfbuzz0b libpangoft2-1.0-0 \
    fonts-amiri fonts-noto-naskh-arabic > /dev/null 2>&1
print("✅ Outils système prêts !")

# --- ÉTAPE 6 : IMPORTS ET VÉRIFICATION ---
import gradio as gr
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML
from PIL import Image

print("✅ Tous les imports sont chargés avec succès.")
print(f"   - Gradio version : {gr.__version__}")
print(f"   - Chemin Gradio  : {gr.__file__}")

🔗 Connexion au Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Dossier projet configuré sur : /content/drive/MyDrive/FreeConvert_Universel_Grandiose
✅ Aucun dossier 'pip_packages' corrompu trouvé sur le Drive.
⏳ Réinstallation forcée de Gradio dans l'environnement Colab...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,

In [9]:
# ==============================================================================
# CORRECTION : Réinstallation des outils système manquants
# ==============================================================================
print("🔧 Réinstallation de LibreOffice, Calibre et ImageMagick...")

# Installation individuelle avec vérification
!apt-get update -qq
!apt-get install -y libreoffice --no-install-recommends
!apt-get install -y calibre --no-install-recommends
!apt-get install -y imagemagick --no-install-recommends

print("\n✅ Vérification post-installation :")
import subprocess

for tool, cmd in [
    ("libreoffice", ["libreoffice", "--version"]),
    ("ebook-convert", ["ebook-convert", "--version"]),
    ("convert", ["convert", "--version"]),
]:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        if result.returncode == 0:
            print(f"   ✅ {tool} — {result.stdout.split(chr(10))[0][:60]}")
        else:
            print(f"   ⚠️ {tool} — installé mais erreur")
    except Exception as e:
        print(f"   ❌ {tool} — toujours manquant : {e}")

🔧 Réinstallation de LibreOffice, Calibre et ImageMagick...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-opensymbol libabsl20220623t64 libabw-0.1-1 libboost-iostreams1.83.0
  libboost-locale1.83.0 libboost-thread1.83.0 libcdr-0.1-1
  libclucene-contribs1t64 libclucene-core1t64 libcolamd3 libe-book-0.1-1
  libeot0 libepubgen-0.1-1 libetonyek-0.1-1 libexttextcat-2.0-0
  libexttextcat-data libfreehand-0.1-1 libgpgme11t64 libgpgmepp6t64
  libgstreamer-plugins-base1.0-0 libharfbuzz-icu0 libhunspell-1.7-0 libhyphen0
  liblangtag-common liblangtag1 libmhash2 libmspub-0.1-1 libmwaw-0.3-3
  libmythes-1.2-0 libodfgen-0.1-1 liborc-0.4-0t64 liborcus-0.18-0
  liborcus-parser-0.18-0 libpagemaker-0.0-0 li

In [11]:
# ==============================================================================
# PARTIE 2 : DÉFINITION DES FORMATS SUPPORTÉS
# ==============================================================================

FORMATS_SUPPORTES = {
    "video": {
        "extensions": ['.mp4', '.mkv', '.avi', '.mov', '.webm', '.flv', '.wmv', '.m4v'],
        "formats_sortie": ["mp4", "mkv", "avi", "webm", "gif", "mp3", "wav"],
        "options": {
            "mp4": ["1080p (Haute qualité)", "720p (Moyenne)", "480p (Légère)"],
            "mkv": ["1080p (Haute qualité)", "720p (Moyenne)", "480p (Légère)"],
            "avi": ["1080p (Haute qualité)", "720p (Moyenne)"],
            "webm": ["1080p (Haute qualité)", "720p (Moyenne)"],
            "gif": ["Original", "480p"],
            "mp3": ["Haute qualité (320kbps)", "Moyenne (192kbps)", "Basse (128kbps)"],
            "wav": ["Standard"]
        }
    },
    "audio": {
        "extensions": ['.mp3', '.wav', '.flac', '.aac', '.ogg', '.m4a', '.wma'],
        "formats_sortie": ["mp3", "wav", "flac", "aac", "ogg"],
        "options": {
            "mp3": ["Haute qualité (320kbps)", "Moyenne (192kbps)", "Basse (128kbps)"],
            "wav": ["Standard"],
            "flac": ["Sans perte"],
            "aac": ["Haute qualité (256kbps)", "Moyenne (128kbps)"],
            "ogg": ["Haute qualité (256kbps)", "Moyenne (128kbps)"]
        }
    },
    "image": {
        "extensions": ['.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tiff', '.heic', '.svg'],
        "formats_sortie": ["png", "jpg", "webp", "bmp", "tiff", "pdf"],
        "options": {
            "png": ["Original", "Haute résolution (300 DPI)", "Basse résolution (72 DPI)"],
            "jpg": ["Original", "Haute qualité (95%)", "Moyenne (80%)", "Basse (60%)"],
            "webp": ["Original", "Haute qualité", "Moyenne"],
            "bmp": ["Original"],
            "tiff": ["Original"],
            "pdf": ["Standard", "Haute qualité"]
        }
    },
    "document": {
        "extensions": ['.docx', '.doc', '.pptx', '.ppt', '.xlsx', '.xls', '.odt', '.txt', '.md', '.html', '.xml', '.py', '.js', '.css'],
        "formats_sortie": ["pdf", "docx", "txt", "html", "md"],
        "options": {
            "pdf": ["Standard A4", "Haute qualité"],
            "docx": ["Standard"],
            "txt": ["Standard"],
            "html": ["Standard"],
            "md": ["Standard"]
        }
    },
    "ebook": {
        "extensions": ['.epub', '.mobi', '.azw3', '.fb2', '.lit'],
        "formats_sortie": ["epub", "mobi", "pdf", "azw3"],
        "options": {
            "epub": ["Standard"],
            "mobi": ["Standard"],
            "pdf": ["Standard A4"],
            "azw3": ["Standard"]
        }
    },
    "archive": {
        "extensions": ['.zip', '.rar', '.7z', '.tar', '.gz'],
        "formats_sortie": ["zip", "tar", "gz"],
        "options": {
            "zip": ["Standard"],
            "tar": ["Standard"],
            "gz": ["Standard"]
        }
    }
}

In [12]:
# ==============================================================================
# PARTIE 3 : FONCTIONS UTILITAIRES
# ==============================================================================

def get_category(ext):
    """Détermine la catégorie d'un fichier selon son extension."""
    ext = ext.lower()
    for cat, data in FORMATS_SUPPORTES.items():
        if ext in data["extensions"]:
            return cat
    return None

def get_output_path(extension, save_mode, original_filename):
    """Génère le chemin de sortie."""
    timestamp = int(time.time())
    base_name = Path(original_filename).stem
    filename = f"{base_name}_{timestamp}.{extension}"

    if save_mode == "💻 Télécharger sur mon PC (Téléchargement direct)":
        return os.path.join(tempfile.gettempdir(), filename)
    else:
        return os.path.join(PROJECT_DIR, filename)

In [13]:
# ==============================================================================
# PARTIE 4 : FONCTIONS DE CONVERSION
# ==============================================================================

def convert_video_audio(input_path, output_path, format_sortie, option):
    """Conversion Vidéo/Audio via FFmpeg."""
    cmd = ['ffmpeg', '-i', input_path, '-y']

    if format_sortie in ['mp4', 'mkv', 'avi', 'webm']:
        if "1080p" in option:
            cmd.extend(['-vf', 'scale=-2:1080'])
        elif "720p" in option:
            cmd.extend(['-vf', 'scale=-2:720'])
        elif "480p" in option:
            cmd.extend(['-vf', 'scale=-2:480'])

    elif format_sortie in ['mp3', 'aac', 'ogg']:
        if "320kbps" in option or "256kbps" in option:
            cmd.extend(['-b:a', '320k'])
        elif "192kbps" in option or "128kbps" in option:
            cmd.extend(['-b:a', '192k'])
        elif "128kbps" in option:
            cmd.extend(['-b:a', '128k'])

    cmd.append(output_path)
    subprocess.run(cmd, check=True, capture_output=True)
    return True

def convert_image(input_path, output_path, format_sortie, option):
    """Conversion d'image via Pillow."""
    img = Image.open(input_path)
    if img.mode in ('RGBA', 'LA', 'P'):
        img = img.convert('RGB')

    if format_sortie == 'pdf':
        img.save(output_path, 'PDF', resolution=300 if "Haute" in option else 72)
    else:
        img.save(output_path, format_sortie.upper())
    return True

def convert_document(input_path, output_path, format_sortie, option):
    """Conversion de document via LibreOffice ou WeasyPrint."""
    ext = Path(input_path).suffix.lower()

    if ext in ['.txt', '.md', '.html', '.xml', '.py', '.js', '.css'] and format_sortie == 'pdf':
        with open(input_path, 'r', encoding='utf-8') as f:
            content = f.read()
        html_content = f"<html><body><pre>{html.escape(content)}</pre></body></html>"
        HTML(string=html_content).write_pdf(output_path)
        return True
    else:
        temp_dir = "/tmp/libreoffice_convert"
        os.makedirs(temp_dir, exist_ok=True)
        cmd = ['libreoffice', '--headless', '--convert-to', format_sortie, '--outdir', temp_dir, input_path]
        subprocess.run(cmd, check=True, capture_output=True)
        pdf_files = list(Path(temp_dir).glob(f'*.{format_sortie}'))
        if pdf_files:
            shutil.move(str(pdf_files[0]), output_path)
            return True
        return False

def convert_ebook(input_path, output_path, format_sortie, option):
    """Conversion d'ebook via Calibre."""
    cmd = ['ebook-convert', input_path, output_path]
    subprocess.run(cmd, check=True, capture_output=True)
    return True

def convert_archive(input_path, output_path, format_sortie, option):
    """Conversion d'archive (extraction/recompression)."""
    temp_extract = "/tmp/archive_extract"
    if os.path.exists(temp_extract):
        shutil.rmtree(temp_extract)
    os.makedirs(temp_extract, exist_ok=True)

    if input_path.endswith('.zip'):
        shutil.unpack_archive(input_path, temp_extract, 'zip')
    elif input_path.endswith('.tar'):
        shutil.unpack_archive(input_path, temp_extract, 'tar')
    elif input_path.endswith('.gz'):
        shutil.unpack_archive(input_path, temp_extract, 'gzip')
    else:
        return False

    if format_sortie == 'zip':
        shutil.make_archive(output_path.replace('.zip', ''), 'zip', temp_extract)
    elif format_sortie == 'tar':
        shutil.make_archive(output_path.replace('.tar', ''), 'tar', temp_extract)
    elif format_sortie == 'gz':
        shutil.make_archive(output_path.replace('.gz', ''), 'gzip', temp_extract)
    return True

In [14]:
# ==============================================================================
# PARTIE 5 : FONCTION PRINCIPALE DE CONVERSION AVEC PROGRESSION + ANNULATION
# ==============================================================================
import threading

# Drapeau global d'annulation (partagé entre les threads Gradio)
CANCEL_FLAG = {"cancel": False}

def request_cancel():
    """Fonction appelée par le bouton d'arrêt."""
    CANCEL_FLAG["cancel"] = True
    return "🛑 Annulation demandée. Le processus va s'arrêter dès que possible..."

def reset_cancel():
    """Réinitialise le drapeau avant chaque nouvelle conversion."""
    CANCEL_FLAG["cancel"] = False

def process_conversion(file, format_sortie, option, save_mode, progress=gr.Progress()):
    """Fonction principale de conversion avec barre de progression et annulation."""
    reset_cancel()  # Réinitialise le drapeau à chaque nouveau lancement

    if file is None:
        return None, "❌ Veuillez sélectionner un fichier."

    progress(0, desc="📥 Analyse du fichier...")
    file_path = file.name
    ext = Path(file_path).suffix.lower()
    category = get_category(ext)

    if category is None:
        return None, f"❌ Format non supporté : {ext}"

    progress(0.1, desc=f"📂 Catégorie détectée : {category.upper()}")

    original_name = Path(file_path).name
    output_path = get_output_path(format_sortie, save_mode, original_name)

    location_msg = "sur votre PC (téléchargeable ci-dessous)" if "PC" in save_mode else f"sur votre Drive ({PROJECT_DIR})"

    try:
        # --- Vérification d'annulation avant conversion ---
        if CANCEL_FLAG["cancel"]:
            raise gr.Error("🛑 Conversion annulée par l'utilisateur (avant conversion).")

        progress(0.3, desc=f"🔄 Conversion en cours vers {format_sortie.upper()}...")

        if category == "video" or category == "audio":
            success = convert_video_audio(file_path, output_path, format_sortie, option)
        elif category == "image":
            success = convert_image(file_path, output_path, format_sortie, option)
        elif category == "document":
            success = convert_document(file_path, output_path, format_sortie, option)
        elif category == "ebook":
            success = convert_ebook(file_path, output_path, format_sortie, option)
        elif category == "archive":
            success = convert_archive(file_path, output_path, format_sortie, option)
        else:
            success = False

        # --- Vérification d'annulation après conversion ---
        if CANCEL_FLAG["cancel"]:
            # Supprime le fichier partiel si annulé
            if os.path.exists(output_path):
                try: os.remove(output_path)
                except: pass
            raise gr.Error("🛑 Conversion annulée par l'utilisateur (après conversion).")

        progress(0.9, desc="💾 Finalisation et sauvegarde...")

        if success and os.path.exists(output_path):
            progress(1.0, desc="✅ Conversion terminée !")
            return output_path, f"✅ Conversion réussie ! Fichier sauvegardé {location_msg}."
        else:
            progress(1.0, desc="❌ Échec")
            return None, "❌ Échec de la conversion. Vérifiez le format ou les options."

    except gr.Error as e:
        # Annulation propre : on affiche le message sans lever d'erreur Python
        progress(1.0, desc="🛑 Annulé")
        return None, str(e)
    except Exception as e:
        progress(1.0, desc="❌ Erreur")
        return None, f"❌ Erreur : {str(e)}"

In [15]:
# ==============================================================================
# PARTIE 6 : FONCTIONS DE MISE À JOUR DYNAMIQUE DE L'INTERFACE
# ==============================================================================

def update_formats(file):
    """Met à jour dynamiquement les formats de sortie selon le fichier uploadé."""
    if file is None:
        return gr.update(choices=[], value=None), gr.update(choices=[], value=None), "Aucun fichier"

    ext = Path(file.name).suffix.lower()
    category = get_category(ext)

    if category is None:
        return gr.update(choices=[], value=None), gr.update(choices=[], value=None), f"❌ Format non supporté : {ext}"

    data = FORMATS_SUPPORTES[category]
    formats = data["formats_sortie"]

    default_format = formats[0]
    options = data["options"].get(default_format, ["Standard"])

    return gr.update(choices=formats, value=default_format), gr.update(choices=options, value=options[0]), f"✅ Catégorie : {category.upper()}"

def update_options(format_sortie, file):
    """Met à jour les options selon le format choisi."""
    if file is None or not format_sortie:
        return gr.update(choices=[], value=None)

    ext = Path(file.name).suffix.lower()
    category = get_category(ext)
    if category is None:
        return gr.update(choices=[], value=None)

    options = FORMATS_SUPPORTES[category]["options"].get(format_sortie, ["Standard"])
    return gr.update(choices=options, value=options[0])

In [16]:
# ==============================================================================
# PARTIE 7.5 : VÉRIFICATION PRÉ-LANCEMENT ET NETTOYAGE DES CACHES
# ==============================================================================
import sys
import os
import shutil
import gc

print("🧹 ÉTAPE 1 : Nettoyage des caches Python...")

# Suppression des fichiers __pycache__ dans le dossier projet
for root, dirs, files in os.walk(PROJECT_DIR):
    for d in list(dirs):
        if d == "__pycache__":
            pycache_path = os.path.join(root, d)
            try:
                shutil.rmtree(pycache_path)
                print(f"   🗑️  Supprimé : {pycache_path}")
            except Exception as e:
                print(f"   ⚠️  Impossible de supprimer {pycache_path} : {e}")

# Forcer le ramasse-miettes
gc.collect()
print("✅ Caches nettoyés et mémoire libérée.")

print("\n🔍 ÉTAPE 2 : Vérification des composants chargés...")

# Liste des éléments qui doivent être définis
required_items = {
    "FORMATS_SUPPORTES": "Dictionnaire des formats (Cellule 3)",
    "get_category": "Fonction utilitaire (Cellule 4)",
    "get_output_path": "Fonction utilitaire (Cellule 4)",
    "convert_video_audio": "Fonction de conversion (Cellule 5)",
    "convert_image": "Fonction de conversion (Cellule 5)",
    "convert_document": "Fonction de conversion (Cellule 5)",
    "convert_ebook": "Fonction de conversion (Cellule 5)",
    "convert_archive": "Fonction de conversion (Cellule 5)",
    "process_conversion": "Fonction principale (Cellule 6)",
    "request_cancel": "Fonction d'annulation (Cellule 6)",
    "update_formats": "Fonction dynamique (Cellule 7)",
    "update_options": "Fonction dynamique (Cellule 7)",
}

all_ok = True
for item, description in required_items.items():
    if item in globals():
        print(f"   ✅ {item} — {description}")
    else:
        print(f"   ❌ {item} MANQUANT — {description}")
        all_ok = False

# Vérification de Gradio
try:
    import gradio as gr
    print(f"\n✅ Gradio est importé (version {gr.__version__})")
    print(f"   Chemin : {gr.__file__}")
    if "/content/drive" in gr.__file__:
        print("   ⚠️  ATTENTION : Gradio est chargé depuis le Drive ! Réexécutez la Cellule 2.")
        all_ok = False
except ImportError:
    print("\n❌ Gradio n'est pas importé. Réexécutez la Cellule 2.")
    all_ok = False

# Vérification des outils système
print("\n🔧 ÉTAPE 3 : Vérification des outils système...")
import subprocess
for tool, check_cmd in [
    ("ffmpeg", "ffmpeg -version"),
    ("libreoffice", "libreoffice --version"),
    ("ebook-convert (calibre)", "ebook-convert --version"),
    ("convert (imagemagick)", "convert --version"),
]:
    try:
        result = subprocess.run(check_cmd.split(), capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            first_line = result.stdout.split("\n")[0][:60]
            print(f"   ✅ {tool} — {first_line}")
        else:
            print(f"   ⚠️  {tool} — installé mais retourne une erreur")
    except Exception as e:
        print(f"   ❌ {tool} — non disponible ({e})")
        all_ok = False

# Résultat final
print("\n" + "="*60)
if all_ok:
    print("🎉 TOUT EST PRÊT ! Vous pouvez lancer la Cellule 8 (Gradio).")
else:
    print("⚠️  Certains éléments manquent. Réexécutez les cellules indiquées ci-dessus.")
print("="*60)

🧹 ÉTAPE 1 : Nettoyage des caches Python...
✅ Caches nettoyés et mémoire libérée.

🔍 ÉTAPE 2 : Vérification des composants chargés...
   ✅ FORMATS_SUPPORTES — Dictionnaire des formats (Cellule 3)
   ✅ get_category — Fonction utilitaire (Cellule 4)
   ✅ get_output_path — Fonction utilitaire (Cellule 4)
   ✅ convert_video_audio — Fonction de conversion (Cellule 5)
   ✅ convert_image — Fonction de conversion (Cellule 5)
   ✅ convert_document — Fonction de conversion (Cellule 5)
   ✅ convert_ebook — Fonction de conversion (Cellule 5)
   ✅ convert_archive — Fonction de conversion (Cellule 5)
   ✅ process_conversion — Fonction principale (Cellule 6)
   ✅ request_cancel — Fonction d'annulation (Cellule 6)
   ✅ update_formats — Fonction dynamique (Cellule 7)
   ✅ update_options — Fonction dynamique (Cellule 7)

✅ Gradio est importé (version 6.30.0)
   Chemin : /usr/local/lib/python3.13/dist-packages/gradio/__init__.py

🔧 ÉTAPE 3 : Vérification des outils système...
   ✅ ffmpeg — ffmpeg version 

In [ ]:
# ==============================================================================
# PARTIE 7 : INTERFACE GRADIO PROFESSIONNELLE ET LANCEMENT
# ==============================================================================

custom_css = """
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background-color: #f8f9fa;
}
.main-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    padding: 20px;
    border-radius: 10px;
    text-align: center;
    margin-bottom: 20px;
}
.card {
    background: white;
    border-radius: 10px;
    padding: 20px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
    margin-bottom: 20px;
}
.btn-primary {
    background: #667eea !important;
    color: white !important;
    border: none !important;
}
.btn-primary:hover {
    background: #5a6fd8 !important;
}
.btn-cancel {
    background: #dc3545 !important;
    color: white !important;
    border: none !important;
}
.btn-cancel:hover {
    background: #c82333 !important;
}
"""

with gr.Blocks(title="FreeConvert Universel Grandiose", theme=gr.themes.Soft(), css=custom_css) as demo:

    gr.HTML("""
    <div class="main-header">
        <h1>🚀 FreeConvert Universel Grandiose</h1>
        <p>Convertissez plus de 1500 formats de fichiers : Vidéo, Audio, Image, Document, Ebook, Archive</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📁 1. Importer votre fichier")
            file_input = gr.File(label="Glissez-déposez ou cliquez pour choisir un fichier")

            status_detect = gr.Textbox(label="📊 Détection", interactive=False, value="En attente...")

            gr.Markdown("### ⚙️ 2. Choisir le format de sortie")
            format_output = gr.Dropdown(label="Format de destination", choices=[], interactive=True)
            option_output = gr.Dropdown(label="Options avancées", choices=[], interactive=True)

            save_location = gr.Radio(
                choices=[
                    "💻 Télécharger sur mon PC (Téléchargement direct)",
                    "📁 Sauvegarder sur Google Drive"
                ],
                value="💻 Télécharger sur mon PC (Téléchargement direct)",
                label="🎯 Emplacement de sauvegarde"
            )

            with gr.Row():
                btn_convert = gr.Button("🔄 Lancer la Conversion", variant="primary", size="lg", scale=2)
                btn_cancel = gr.Button("🛑 Arrêter", variant="stop", size="lg", scale=1)

        with gr.Column(scale=1):
            gr.Markdown("### 📥 3. Résultat")
            progress_bar = gr.Progress()
            output_file = gr.File(label="Fichier converti")
            status_message = gr.Textbox(label="Message de statut", interactive=False, lines=3)
            cancel_status = gr.Textbox(label="Statut d'annulation", interactive=False, value="")

            gr.Markdown("""
            ---
            ### 💡 Instructions
            1. Importez un fichier (ex: `.mp4`, `.pdf`, `.png`).
            2. Le système détecte automatiquement la catégorie et propose les formats.
            3. Choisissez le format et les options (résolution, qualité).
            4. Cliquez sur "Lancer la Conversion".
            5. En cas de besoin, cliquez sur "🛑 Arrêter" pour annuler.
            6. Téléchargez le résultat ou retrouvez-le sur votre Drive.
            """)

    # Événements
    file_input.change(
        fn=update_formats,
        inputs=[file_input],
        outputs=[format_output, option_output, status_detect]
    )

    format_output.change(
        fn=update_options,
        inputs=[format_output, file_input],
        outputs=[option_output]
    )

    btn_convert.click(
        fn=process_conversion,
        inputs=[file_input, format_output, option_output, save_location],
        outputs=[output_file, status_message],
        show_progress="full"
    )

    btn_cancel.click(
        fn=request_cancel,
        inputs=[],
        outputs=[cancel_status]
    )

    gr.Markdown("""
    ---
    ### 🛠️ Technologies utilisées
    - **Vidéo/Audio** : FFmpeg
    - **Documents** : LibreOffice & WeasyPrint
    - **Ebooks** : Calibre
    - **Images** : Pillow & ImageMagick
    - **Archives** : Python standard library
    """)

demo.launch(share=True, debug=True)

/tmp/ipykernel_9594/2062915255.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="FreeConvert Universel Grandiose", theme=gr.themes.Soft(), css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9c67ed04e9c99e39bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
